# Week 4 — 策略報酬的統計推論

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 估計平均報酬的標準誤與信賴區間。
- 用 bootstrap 建立平均報酬的信賴區間。
- 理解 p-value 的意義與常見誤用。
- 示範「測試大量隨機策略」如何製造假陽性。

## 預估學習時間

約 9–11 小時。

## 先備概念

- Week 3 的抽樣不確定性
- 平均、標準差

## 外部學習資源

- [MIT OpenCourseWare 18.05 Introduction to Probability and Statistics](https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/)
- [NTU OpenCourseWare 統計學一上與計量導論](https://ocw.aca.ntu.edu.tw/courses/112S103)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

In [ ]:
# 教學樣式設定（CJK 字型、負號正常顯示、固定隨機種子）
import matplotlib as _mpl
_mpl.rcParams['font.sans-serif'] = [
    'PingFang TC', 'Heiti TC', 'Microsoft JhengHei',
    'Noto Sans CJK TC', 'Noto Sans TC',
    'WenQuanYi Zen Hei', 'Source Han Sans TC',
    'Arial Unicode MS', 'DejaVu Sans',
]
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## 概念說明

### 估計量、標準誤與信賴區間

**估計量**是資料的函數（例如樣本平均）。它有 **bias**（系統性偏差）與 **variance**（隨樣本變動）。樣本平均的**標準誤**為 $s/\sqrt n$。

**信賴區間**給出「與資料相容的參數範圍」。95% 信賴區間的正確解讀是：若重複抽樣很多次，約 95% 的區間會涵蓋真實參數。

### p-value 與其誤用

p-value 是「**若虛無假設為真**，看到目前或更極端結果的機率」。它**不是**「策略有效的機率」。最危險的誤用是 **multiple testing**：測試夠多隨機策略，總會有幾個僅憑運氣就「顯著」。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.math.probability import simulate_normal
from quant_math_roadmap.math.statistics import (
    bootstrap_mean_ci, confidence_interval_mean,
    false_discovery_demo, one_sample_ttest, standard_error_of_mean,
)
from quant_math_roadmap.finance.metrics import sharpe_ratio
import pandas as pd

### 平均報酬的標準誤與信賴區間

In [ ]:
returns = simulate_normal(mean=0.0004, std=0.012, size=252, seed=42)
se = standard_error_of_mean(returns)
lower, upper = confidence_interval_mean(returns, confidence=0.95)
print(f'樣本平均日報酬 = {returns.mean():.6f}')
print(f'標準誤 = {se:.6f}')
print(f'95% 信賴區間 = [{lower:.6f}, {upper:.6f}]')
print('注意：信賴區間很可能涵蓋 0 — 我們無法排除「真實期望為 0」。')

### Bootstrap 信賴區間

In [ ]:
boot_lower, boot_upper = bootstrap_mean_ci(
    returns, confidence=0.95, n_resamples=5000, seed=0)
print(f'bootstrap 95% 信賴區間 = [{boot_lower:.6f}, {boot_upper:.6f}]')
print(f't 分布   95% 信賴區間 = [{lower:.6f}, {upper:.6f}]')
print('兩種方法給出相近的區間；bootstrap 不需要常態假設。')

### Block bootstrap：當報酬有自相關時

普通 bootstrap 把每個觀測值當成獨立可重抽——這隱含 i.i.d. 假設。若報酬有**自相關**（動能型策略的報酬常有），普通 bootstrap 會**低估**平均報酬的不確定性。**circular block bootstrap** 改成整塊連續區段重抽，保留區塊內的相依結構。

下面用一個高自相關的 AR(1) 序列示範兩種方法的差距。

In [ ]:
from quant_math_roadmap.math.statistics import block_bootstrap_mean_ci
from quant_math_roadmap.data import generate_ar1_series

# phi=0.9 的 AR(1)：有效樣本數遠小於名目樣本數
persistent = generate_ar1_series(2000, phi=0.9, seed=7).to_numpy()
plain_ci = bootstrap_mean_ci(persistent, seed=0)
block_ci = block_bootstrap_mean_ci(persistent, block_size=50, seed=0)
print(f'普通 bootstrap  95% CI 寬度 = {plain_ci[1] - plain_ci[0]:.4f}')
print(f'block bootstrap 95% CI 寬度 = {block_ci[1] - block_ci[0]:.4f}')
print('自相關資料下，普通 bootstrap 給出過窄（過度自信）的區間。')

### 比較兩個合成策略

In [ ]:
strategy_a = simulate_normal(mean=0.0002, std=0.010, size=252, seed=1)
strategy_b = simulate_normal(mean=0.0007, std=0.018, size=252, seed=2)
for name, s in [('策略 A', strategy_a), ('策略 B', strategy_b)]:
    t = one_sample_ttest(s, popmean=0.0)
    ci = confidence_interval_mean(s)
    print(f'{name}: 平均={s.mean():.6f}, p-value={t.p_value:.3f}, '
          f'95% CI=[{ci[0]:.6f}, {ci[1]:.6f}]')

即使某個策略的樣本平均較高，其 p-value 仍可能不顯著、信賴區間仍涵蓋 0。**較高的歷史平均報酬不等於較高的真實期望報酬。**

### 多重檢定：假陽性的製造機

下面產生大量**純雜訊**策略（真實期望報酬皆為 0），看看有多少會在$\alpha=0.05$ 下被誤判為「顯著」。

In [ ]:
demo = false_discovery_demo(n_strategies=500, n_periods=252,
                            alpha=0.05, seed=0)
for k, v in demo.items():
    print(f'{k}: {v}')
print()
print('全部 500 檔策略都是純雜訊，所有「顯著」結果都是假陽性。')

In [ ]:
# 視覺化：最佳雜訊策略的權益曲線看起來也可能很漂亮
rng = np.random.default_rng(0)
noise = rng.standard_normal((500, 252)) * 0.01
totals = (1 + noise).prod(axis=1) - 1
best = noise[int(np.argmax(totals))]
equity = (1 + pd.Series(best)).cumprod()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(equity.index, equity.values, label='500 檔雜訊策略中「最好」的一檔')
ax.axhline(1.0, linestyle='--', label='起始資金')
ax.set_title('一條漂亮的權益曲線 — 但它純粹是運氣')
ax.set_xlabel('交易日')
ax.set_ylabel('權益（起始 = 1）')
ax.legend()
plt.show()

這條曲線完全由雜訊產生，卻可能比真正有訊號的策略還漂亮。**一條漂亮的權益曲線無法證明策略有效。**

### 把「多重檢定」變成數字：PSR 與 Deflated Sharpe Ratio

前面示範了「測試夠多策略，總有幾個看起來顯著」。Bailey 與 López de Prado 把這個警告變成可計算的指標：

- **PSR（Probabilistic Sharpe Ratio）**：考慮樣本長度、偏態與峰度後，「真實 Sharpe 大於基準」的機率；
- **DSR（Deflated Sharpe Ratio）**：把基準從 0 提高到「N 個無技能策略中最幸運者的期望 Sharpe」——你試過越多策略，入選者要跨過的門檻就越高。

In [ ]:
from quant_math_roadmap.finance.metrics import (
    deflated_sharpe_ratio, expected_max_sharpe, probabilistic_sharpe_ratio,
)

# 用前面那批 500 檔純雜訊策略：挑出總報酬最高的「冠軍」
best_returns = pd.Series(best)

# 各策略的每期 Sharpe 估計值，其跨策略標準差用於期望最大值公式
per_period_sr = noise.mean(axis=1) / noise.std(axis=1, ddof=1)
sr_std = float(per_period_sr.std(ddof=1))

psr = probabilistic_sharpe_ratio(best_returns)
benchmark = expected_max_sharpe(500, sr_std=sr_std)
dsr = deflated_sharpe_ratio(best_returns, n_trials=500, sr_std=sr_std)
print(f'冠軍策略的 PSR（基準 SR=0）  = {psr:.4f}  <- 看起來頗有把握')
print(f'500 試誤下的期望最大 SR      = {benchmark:.4f}')
print(f'冠軍策略的 DSR（扣除選擇效應）= {dsr:.4f}  <- 原形畢露')

PSR 看起來很高——但那只是因為我們**挑了最幸運的一檔**。把「試了 500 次」誠實地放進基準後，DSR 立刻塌回去：這檔策略的「優異」表現與純運氣無法區分。**報告回測結果時，必須一併報告你總共試了多少組合。**

### 風險調整指標的警語

In [ ]:
sr = sharpe_ratio(pd.Series(returns), frequency='daily')
print(f'年化 Sharpe ratio = {sr:.3f}')
print('警語：Sharpe 是估計值，本身有抽樣誤差；它忽略偏態與厚尾；')
print('短樣本下，回測 Sharpe 為 2 也可能與「真實 Sharpe 為 0」相容。')

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用一句話寫出 p-value 的正確定義。
2. 「95% 信賴區間」正確的解讀是什麼？常見的錯誤解讀又是什麼？
3. 為什麼「測試很多策略後挑出最好的一個」會讓 p-value 失去意義？

### 應用練習

In [ ]:
# 應用練習 1：對 strategy_a 做 bootstrap，比較 90% 與 99% 信賴區間的寬度。
ci90 = None  # TODO: bootstrap_mean_ci(strategy_a, confidence=0.90, seed=0)
ci99 = None  # TODO: bootstrap_mean_ci(strategy_a, confidence=0.99, seed=0)
if ci90 and ci99:
    print('90% 寬度:', ci90[1] - ci90[0])
    print('99% 寬度:', ci99[1] - ci99[0])

In [ ]:
# 應用練習 2：把 false_discovery_demo 的 alpha 改成 0.01，
# 觀察假陽性數量如何變化。
strict = None  # TODO: false_discovery_demo(n_strategies=500, alpha=0.01, seed=0)
if strict is not None:
    print(strict)

### 反思問題

1. 你在大量參數組合中找到一個回測表現很好的策略。在相信它之前，你應該對「多重檢定」與「樣本外驗證」做哪些事？

## 小測驗（自我檢核）
回答下面的選擇題，然後執行下一格自動對答案。答案以雜湊儲存，不會直接洩漏。

**Q1. p-value 的正確定義是？**
- A. 虛無假設為真的機率
- B. 在虛無假設為真的前提下，看到目前或更極端結果的機率
- C. 策略有效的機率
- D. 犯第一類錯誤的機率

**Q2. 95% 信賴區間的正確解讀是？**
- A. 參數有 95% 機率落在區間內
- B. 重複抽樣下，約 95% 的區間會涵蓋真實參數
- C. 95% 的資料落在區間內
- D. 預測準確率為 95%

**Q3. 測試 100 個純雜訊策略、顯著水準 α=0.05，期望有幾個「顯著」？**
- A. 0 個
- B. 1 個
- C. 5 個
- D. 50 個

**Q4. block bootstrap 相對普通 bootstrap 的目的為何？**
- A. 計算更快
- B. 保留資料的自相關結構
- C. 讓樣本更大
- D. 降低變異數

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: 填入 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '304c8f72060c041b', 2: 'd62e543599a0a653', 3: 'af0be7d09fe40e70', 4: '83613d5a2b2977f8'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: 未作答')
        continue
    _h = _hashlib.sha256(f'qmr-w4-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ 正確' if _ok else '✘ 不正確'))
print(f'得分: {_n_correct} / {len(my_answers)}')

## 常見錯誤

- **把 p-value 解讀成「策略有效的機率」。**
- **測試大量策略後只報告最好的，卻不做多重檢定校正。**
- **把統計顯著當成經濟顯著（即使有效也可能被成本吃掉）。**
- **用單一回測 Sharpe ratio 就下結論，忽略它的抽樣誤差。**

## 完成本週後，你應該能做到什麼

- [ ] 能計算並解讀平均報酬的信賴區間（t 與 bootstrap）。
- [ ] 能正確說明 p-value 的意義。
- [ ] 能示範並解釋多重檢定造成的假陽性。
- [ ] 能說出 Sharpe ratio 的至少三項侷限。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../docs/math/) 與 [`docs/finance/`](../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。